# TinyYOLO Experiment Suite — Notebook 3/7
## Cross-Ablation: Activation × Attention (New Table for Reviewer C1)

**CRITICAL EXPERIMENT** — resolves the FP32 accuracy inversion.

| Config | Activation | Attention | Expected |
|--------|-----------|-----------|----------|
| A | SiLU (standard) | SE+Spatial (default) | ~38.7% |
| B | SiLU (standard) | ECA | NEW |
| C | ReLU6 (quantized) | SE+Spatial | NEW |
| D | ReLU6 (quantized) | ECA (default) | ~41.2% |

**GPU Time:** ~63h T4 (12 runs × ~5.2h). For Kaggle, set `RUN_CONFIGS` below.

**Results → `experiments/results/xabl-*/summary.json`**

In [ ]:
#@title ⚙️ Configuration — Edit this cell

FAST_MODE = True  #@param {type:"boolean"}
RUN_CONFIGS = ['A', 'B', 'C', 'D']  #@param

if FAST_MODE:
    print("🚀 Running in FAST_MODE (Rapid Verification)")
    SEEDS = [42]                 # Single seed instead of 3
    EPOCHS = 100                 # Early convergence with pre-training
    IMGSZ = 320                  # 40% reduction in conv FLOPs
    BATCH = 32
else:
    print("📚 Running in FULL PUBLICATION RIGOR mode")
    SEEDS = [42, 123, 256]
    EPOCHS = 300
    IMGSZ = 416
    BATCH = 32

REPO = 'https://github.com/ShMazumder/tinyYOLO.git'

CONFIGS = {
    'A': {'variant': 'standard',  'attention': 'default', 'label': 'SiLU+SE'},
    'B': {'variant': 'standard',  'attention': 'eca',     'label': 'SiLU+ECA'},
    'C': {'variant': 'quantized', 'attention': 'se',      'label': 'ReLU6+SE'},
    'D': {'variant': 'quantized', 'attention': 'default', 'label': 'ReLU6+ECA'},
}

In [ ]:
import os, sys, socket, shutil
from pathlib import Path
try: socket.create_connection(('github.com', 80), timeout=5)
except: print('❌ No internet!'); sys.exit(1)
if Path('tinyYOLO').exists(): shutil.rmtree('tinyYOLO')
!git clone {REPO} --depth 1
%cd tinyYOLO
!pip install -e . -q 2>&1 | tail -1
!pip install tqdm timm psutil -q 2>&1 | tail -1
import torch
if torch.cuda.is_available(): print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')" 2>&1 | tail -3

# Verify all 4 configs build
from tinyYOLO.models import build_model
x = torch.randn(1,3,416,416)
for k, c in CONFIGS.items():
    m, i = build_model(task='det', variant=c['variant'], nc=20, attention=c['attention'])
    _ = m(x)
    print(f"  ✓ Config {k} ({c['label']}): {i['total_params_M']}M, attn3={m.backbone.attn3.__class__.__name__}")
print('✅ All configs verified!')

In [ ]:
import time
total = len(RUN_CONFIGS) * len(SEEDS)
run_i = 0
for cfg_key in RUN_CONFIGS:
    cfg = CONFIGS[cfg_key]
    for seed in SEEDS:
        run_i += 1
        name = f"xabl-{cfg_key}-{cfg['label'].lower().replace('+','-')}-seed{seed}"
        if Path(f'experiments/results/{name}/summary.json').exists():
            print(f'⏭️  [{run_i}/{total}] {name} — done'); continue
        print(f"\n{'='*60}\n🏃 [{run_i}/{total}] Config {cfg_key} ({cfg['label']}) seed={seed}\n{'='*60}")
        t0 = time.time()
        !python scripts/train.py --task det --variant {cfg['variant']} --attention {cfg['attention']} --lr 2e-4 --amp --cache --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 150 \
            --data voc.yaml --imgsz {IMGSZ} --epochs {EPOCHS} --batch {BATCH} --seed {seed} \
            --warmup 3 --pretrained --compile --name {name}
        print(f'⏱️  {(time.time()-t0)/3600:.1f}h')
print(f'\n✅ Cross-ablation complete!')

In [ ]:
import json, glob
import numpy as np

results = {}
for k in ['A','B','C','D']:
    cfg = CONFIGS[k]
    pattern = f"xabl-{k}-{cfg['label'].lower().replace('+','-')}-seed*"
    maps = []
    for d in sorted(glob.glob(f'experiments/results/{pattern}')):
        s = Path(d) / 'summary.json'
        if s.exists():
            with open(s) as f: data = json.load(f)
            maps.append(data.get('best_map50', 0) * 100)
    results[f"{k} ({cfg['label']})"] = maps

print('='*70)
print('CROSS-ABLATION: Activation × Attention')
print('='*70)
print(f'{"Config":<20} {"Runs":>5} {"mAP@50":>10} {"Std":>8}')
print('-'*70)
for cfg, maps in results.items():
    if maps: print(f'{cfg:<20} {len(maps):>5} {np.mean(maps):>10.1f} {np.std(maps):>8.2f}')
    else: print(f'{cfg:<20}   N/A')
print('='*70)

vals = list(results.values())
if all(len(v) >= 2 for v in vals):
    a,b,c,d = [np.mean(v) for v in vals]
    eca_eff = ((b-a)+(d-c))/2
    relu6_eff = ((c-a)+(d-b))/2
    inter = (d-c)-(b-a)
    print(f'\n📊 Factorial Analysis:')
    print(f'  ECA effect:         {eca_eff:+.1f}%')
    print(f'  ReLU6 effect:       {relu6_eff:+.1f}%')
    print(f'  Interaction:        {inter:+.1f}%')
    print(f'  Gap explained:      {eca_eff+relu6_eff+inter:+.1f}% (observed: {d-a:+.1f}%)')
    try:
        from scipy import stats
        t,p = stats.ttest_ind(vals[3], vals[0])
        print(f'  T-test D vs A:      t={t:.2f}, p={p:.4f} ({"SIG" if p<0.05 else "n.s."} at p<0.05)')
    except: pass

out = {'experiment':'cross_ablation', 'results':{k:{'maps':v,'mean':float(np.mean(v)),'std':float(np.std(v))} for k,v in results.items() if v}}
with open('experiments/results/cross_ablation_summary.json','w') as f: json.dump(out,f,indent=2)
print('\nSaved: experiments/results/cross_ablation_summary.json')